In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists())

# Source files: outputs from the FinBERT and LDA modules
FINBERT_PATH = project_root / 'data' / 'processed' / 'firm_finbert_features.csv'
LDA_PATH     = project_root / 'data' / 'processed' / 'firm_lda_features.csv'

print(f"FinBERT path: {FINBERT_PATH.exists()} -> {FINBERT_PATH}")
print(f"LDA path:     {LDA_PATH.exists()} -> {LDA_PATH}")

finbert = pd.read_csv(FINBERT_PATH)
lda     = pd.read_csv(LDA_PATH)

print(f"\nFinBERT features: {finbert.shape}")
print(f"  Columns: {finbert.columns.tolist()}")
print(f"\nLDA features: {lda.shape}")
print(f"  Columns: {lda.columns.tolist()}")

FinBERT path: True -> /Users/harrymac/Desktop/Columbia S2/Machine Learning2/Group/deepseek-event-study-ml/data/processed/firm_finbert_features.csv
LDA path:     True -> /Users/harrymac/Desktop/Columbia S2/Machine Learning2/Group/deepseek-event-study-ml/data/processed/firm_lda_features.csv

FinBERT features: (400, 13)
  Columns: ['companyid', 'companyname', 'n_sentences', 'mean_p_positive', 'mean_p_negative', 'mean_p_neutral', 'mean_sentiment', 'std_sentiment', 'pct_negative', 'pct_neutral', 'pct_positive', 'ratio_strong_negative', 'ratio_strong_positive']

LDA features: (400, 17)
  Columns: ['companyid', 'companyname', 'topic_00_energy_utilities', 'topic_01_advertising_marketing', 'topic_02_ai_semiconductor_design', 'topic_03_ai_enterprise_healthcare_services', 'topic_04_financial_reporting_geographic', 'topic_05_financial_reporting_accounting', 'topic_06_government_defense', 'topic_07_project_execution_capital', 'topic_08_china_ai_semiconductor_supply_chain', 'topic_09_ai_cloud_infras

In [3]:
# Verify both dataframes cover the same firms
finbert_ids = set(finbert['companyid'])
lda_ids     = set(lda['companyid'])

both     = finbert_ids & lda_ids
only_fb  = finbert_ids - lda_ids
only_lda = lda_ids - finbert_ids

print(f"Firms in both datasets:    {len(both)}")
print(f"Firms only in FinBERT:     {len(only_fb)}")
print(f"Firms only in LDA:         {len(only_lda)}")

if only_fb:
    print(f"  FinBERT-only IDs: {list(only_fb)[:5]}")
if only_lda:
    print(f"  LDA-only IDs: {list(only_lda)[:5]}")

# We expect perfect overlap since both were built from the same input
assert len(only_fb) == 0 and len(only_lda) == 0, \
    "Firm lists don't match - investigate before merging"
print("\nBoth datasets cover identical firms. Safe to inner-merge.")

Firms in both datasets:    400
Firms only in FinBERT:     0
Firms only in LDA:         0

Both datasets cover identical firms. Safe to inner-merge.


In [5]:
X_text = finbert.merge(lda, on=['companyid', 'companyname'], how='inner')

print(f"Merged shape: {X_text.shape}")
print(f"\nAll columns:")
for i, c in enumerate(X_text.columns):
    print(f"  {i:2d}. {c}")

Merged shape: (400, 28)

All columns:
   0. companyid
   1. companyname
   2. n_sentences
   3. mean_p_positive
   4. mean_p_negative
   5. mean_p_neutral
   6. mean_sentiment
   7. std_sentiment
   8. pct_negative
   9. pct_neutral
  10. pct_positive
  11. ratio_strong_negative
  12. ratio_strong_positive
  13. topic_00_energy_utilities
  14. topic_01_advertising_marketing
  15. topic_02_ai_semiconductor_design
  16. topic_03_ai_enterprise_healthcare_services
  17. topic_04_financial_reporting_geographic
  18. topic_05_financial_reporting_accounting
  19. topic_06_government_defense
  20. topic_07_project_execution_capital
  21. topic_08_china_ai_semiconductor_supply_chain
  22. topic_09_ai_cloud_infrastructure
  23. topic_10_chinese_gaming_streaming
  24. topic_11_ai_saas_cybersecurity
  25. ai_topic_loading_sum
  26. ai_keyword_count
  27. ai_keyword_ratio


In [7]:
# Identify highly correlated feature pairs so downstream (D) is aware
# which variables to drop when fitting linear models
print("=" * 70)
print("Highly correlated feature pairs (|r| > 0.9)")
print("=" * 70)

feature_cols = finbert_cols + lda_cols
corr_matrix = X_text[feature_cols].corr()

# Find upper-triangle pairs with abs(corr) > 0.9
high_corr_pairs = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.9:
            high_corr_pairs.append((feature_cols[i], feature_cols[j], r))

# Sort by absolute correlation, descending
high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

if high_corr_pairs:
    print(f"Found {len(high_corr_pairs)} pairs with |correlation| > 0.9:\n")
    for a, b, r in high_corr_pairs:
        print(f"  r = {r:+.3f}  |  {a}  <-->  {b}")
else:
    print("No feature pairs with |correlation| > 0.9.")

Highly correlated feature pairs (|r| > 0.9)
Found 8 pairs with |correlation| > 0.9:

  r = +0.999  |  pct_positive  <-->  ratio_strong_positive
  r = +0.997  |  pct_negative  <-->  ratio_strong_negative
  r = +0.993  |  mean_p_neutral  <-->  pct_neutral
  r = +0.992  |  mean_p_positive  <-->  pct_positive
  r = +0.991  |  mean_p_positive  <-->  ratio_strong_positive
  r = +0.989  |  mean_p_negative  <-->  pct_negative
  r = +0.989  |  mean_p_negative  <-->  ratio_strong_negative
  r = +0.954  |  ai_keyword_count  <-->  ai_keyword_ratio


In [8]:
OUTPUT_LOCAL  = Path('X_text.csv')
OUTPUT_SHARED = project_root / 'data' / 'processed' / 'X_text.csv'

X_text.to_csv(OUTPUT_LOCAL, index=False)
X_text.to_csv(OUTPUT_SHARED, index=False)
